# Track 1: Quantum Simulation of Photo-Induced Charge Dynamics ( From Ethylene to Photosynthesis )

## Task 1: Constructing the Ethylene Hamiltonian and Mapping to Qubits

### Ethylene Geometry at Equilibrium

```

 H       H 
  \     /  
   C = C   
  /     \  
 H       H 

```
C=C Bond Length $\approx 1.34Å$ 

C-H Bond Length $\approx 1.087Å$

H-C-H Bond Angle $\approx 117.3^{\circ}$

Carbons are relatively positioned at $ x=0, y=0 $ and aligned along z-axis where the distance of each from the origin is equal in value.

Carbon $ |d_z| = \frac{1.34}{2}Å = 0.67Å $

At $\pm0.67$ on the z-axis where the carbon atoms are located, bonds with the hydrogen atoms form. These bonds are tilted by an angle. We calculate the displacements on $y, z$ axes.

Hydrogen $|d_y| = 1.087\sin(\frac{117.3}{2}) = 0.928Å$

Hydrogen $|d_z| = 1.087\cos(\frac{117.3}{2}) + $ Carbon $|d_z| = 0.5655Å + 0.67Å = 1.2355Å$

Meanwhile there is no displacement on the x axis because it is planar.

Now we can finally construct the PySCF driver for Ethylene at equilibrium as follows:

In [69]:
from qiskit_nature.second_q.drivers import PySCFDriver

eth_mol = PySCFDriver(
    atom="C 0 0 0.67; H 0 0.928 1.2355; H 0 -0.928 1.2355; C 0 0 -0.67; H 0 0.928 -1.2355; H 0 -0.928 -1.2355",
    basis='sto3g'
)

es_problem = eth_mol.run()

### Ethylene Geometry at a Twisted $90^{\circ}$ Angle ($CH_2$ Groups Perpendicular to Eachother)

```
  H H     
  |  \    
  C - C   
  |    \  
  H     H 
```

Twisting one of the $CH_2$ groups $90^\circ$ isn't merely a change in the axes,  such alteration eliminates the spatial overlap which breaks the  $\pi$ bond into a relatively longer $\sigma$ bond. It also causes change in the C-H bond length and the bond angle because of the general change in the electronic structure.

The approximated new lengths and angle are as follows:

C-C Bond Length $\approx 1.46Å$ 

C-H Bond Length $\approx 1.08Å$

H-C-H Bond Angle $\approx 121^{\circ}$

Carbon $|d_z| = 1.46/2 = 0.73Å$

But now after the twist, hydrogens are on the x-axis instead of y where the new displacement on the x-axis is equal to the one on the y-axis at equilibrium, while remaining on the z-axis as well

Hydrogen $|d_x| = 0.928$

Hydrogen $|d_z| = 1.08\cos(\frac{121}{2}) +$ Carbon $ |d_z| = 0.53Å + 0.73Å = 1.26Å$



In [70]:
eth_mol_twisted = PySCFDriver(
    atom="C 0 0 0.73; H 0.928 0 1.26; H -0.928 0 1.26; C 0 0 -0.73; H 0.928 0 -1.26; H -0.928 0 -1.26",
    basis='sto3g'
)

es_problem_twisted = eth_mol_twisted.run()

### Reducing to Active Space

2 electrons, 2 orbitals expandable to 2 electrons, 4 orbitals

In [71]:
n_e = 2
n_orb = 2 # change to 4 for expansion

from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

transformer = ActiveSpaceTransformer(num_electrons=n_e,num_spatial_orbitals=n_orb)

reduced_problem = transformer.transform(es_problem)
reduced_problem_twisted = transformer.transform(es_problem_twisted)

### Mapping to Qubits and Getting Hamiltonian for Ethylene in Equilibrium and Twisted Geometry

In [72]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()
h = mapper.map(reduced_problem.second_q_ops()[0]) # index of the main hamiltonian in returned tuple
h_twisted = mapper.map(reduced_problem_twisted.second_q_ops()[0])

In [73]:
print(h)
print(h_twisted)

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.67810865+0.j,  0.07683157+0.j, -0.07802605+0.j,  0.07683157+0.j,
 -0.07802605+0.j,  0.08418098+0.j,  0.12685108+0.j,  0.12720766+0.j,
  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,
  0.12720766+0.j,  0.13086927+0.j,  0.08418098+0.j])
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.65809122+0.j,  0.06206056+0.j, -0.0626729 +0.j,  0.06206056+0.j,
 -0.0626729 +0.j,  0.07940855+0.j,  0.12362153+0.j,  0.12451616+0.j,
  0.04510761+0.j,  0.04510761+0.j,  0.04510761+0.j,  0.04510761+0.j,
  0.12451616+0.j,  0.12784495+0.j,  0.07940855+0.j])


**Now that we constructed the ethylene Hamiltonian in both geometries, for the next tasks we will only focus on ethylene at equilibrium.**

## Task 2: Ground State VQE

### Bulding UCCSD Ansatz

In order to build a UCCSD ansatz we must start from a Hartree-Fock initial state

In [74]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

num_spatial_orbitals = reduced_problem.num_spatial_orbitals
num_particles = reduced_problem.num_particles

initial_state = HartreeFock(num_spatial_orbitals, num_particles, mapper)

uccsd_ansatz = UCCSD(num_spatial_orbitals, num_particles, mapper, initial_state = initial_state)

### Running VQE for UCCSD Ansatz

In [75]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector

estimator = StatevectorEstimator()

uccsd_vqe = VQE(estimator=estimator, ansatz=uccsd_ansatz, optimizer=COBYLA())

uccsd_result = uccsd_vqe.compute_minimum_eigenvalue(operator=h)

optimal_uccsd = uccsd_ansatz.assign_parameters(uccsd_result.optimal_parameters)

uccsd_gse = uccsd_result.eigenvalue.real # UCCSD ground state energy

uccsd_gsv = Statevector(optimal_uccsd) # UCCSD ground state vector

/home/rabee/.qiskit/lib/python3.14/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/home/rabee/.qiskit/lib/python3.14/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


### Building Hardware Efficient Ansatz

In [76]:
from qiskit.circuit.library import efficient_su2

num_qubits = h.num_qubits # = 2 * num_spatial_orbitals

hea = efficient_su2(num_qubits)

### Running VQE for HEA

In [77]:
hea_vqe = VQE(estimator=estimator, ansatz=hea, optimizer=COBYLA())

hea_result = hea_vqe.compute_minimum_eigenvalue(operator=h)

optimal_hea = hea.assign_parameters(hea_result.optimal_parameters)

hea_gse = hea_result.eigenvalue.real # HEA ground state energy

hea_gsv = Statevector(optimal_hea) # HEA ground state vector